Team Members:
- Liad Malachi: 212452882
- Shirel Elmaleh: 345653984
- Github: https://github.com/Shirelmaleh/Project-part1-LIad-and-Shirel 

# Data Analysis Project - Movies

In [15]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import urllib.parse
import time
import re
import numpy as np
from IPython.display import display

Loading the 'basics' dataset and filtering to keep only the relevant movies.


In [2]:
print("Initializing Data Sources...")
data_links = {
    'basics': "https://datasets.imdbws.com/title.basics.tsv.gz",
    'ratings': "https://datasets.imdbws.com/title.ratings.tsv.gz",
    'principals': "https://datasets.imdbws.com/title.principals.tsv.gz"
}

print("Loading and filtering base titles in chunks (Memory Safe)...")

# Define only the columns we actually need to save memory
needed_cols = ['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']

# Efficient loading using chunks, usecols, and dtype=str to avoid mixed-types warnings
chunk_iterator = pd.read_csv(data_links['basics'], 
                             sep='\t', 
                             na_values='\\N', 
                             compression='gzip', 
                             usecols=needed_cols, 
                             chunksize=500000, 
                             dtype=str)

regex_filter = r'(?i)^T[a-i]'
filtered_chunks = []

for chunk in chunk_iterator:
    # 1. Filter for movies
    movies_subset = chunk[chunk['titleType'] == 'movie'].copy()
    
    if movies_subset.empty:
        continue
        
    # 2. Filter by Year
    movies_subset['startYear'] = pd.to_numeric(movies_subset['startYear'], errors='coerce')
    movies_subset = movies_subset[movies_subset['startYear'] <= 2024]
    
    # 3. Filter by Runtime
    movies_subset['runtimeMinutes'] = pd.to_numeric(movies_subset['runtimeMinutes'], errors='coerce')
    movies_subset = movies_subset[(movies_subset['runtimeMinutes'] >= 60) & (movies_subset['runtimeMinutes'] <= 300)]
    
    # 4. Filter by REGEX on primaryTitle
    movies_subset = movies_subset[movies_subset['primaryTitle'].str.contains(regex_filter, regex=True, na=False)]
    
    # Append the filtered chunk to our list
    filtered_chunks.append(movies_subset)

# Concatenate all the small, filtered chunks back into a single DataFrame
target_movies_df = pd.concat(filtered_chunks, ignore_index=True)

# Process genres
target_movies_df['genres'] = target_movies_df['genres'].str.split(',')
target_movies_df = target_movies_df[['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres']]

print(f"Total valid movies after Ta-Ti filter: {len(target_movies_df)}")

Initializing Data Sources...
Loading and filtering base titles in chunks (Memory Safe)...
Total valid movies after Ta-Ti filter: 55892


Merging the 'ratings' table and selecting the top 5,000 most popular movies by vote count.
Merging the 'principals' table and inserting the actors ID numbers.

In [3]:
print("Merging ratings...")
scores_df = pd.read_csv(data_links['ratings'], sep='\t', na_values='\\N', compression='gzip')
target_movies_df = pd.merge(target_movies_df, scores_df, on='tconst', how='left')

target_movies_df = target_movies_df.sort_values(by='numVotes', ascending=False).head(5000)

print(f"Sorting complete. Taking top {len(target_movies_df)} movies for processing.")
print("Fetching lead actors in chunks (Memory Safe)...")

cast_reader = pd.read_csv(data_links['principals'], sep='\t', na_values='\\N', compression='gzip', usecols=['tconst', 'ordering', 'nconst', 'category'], chunksize=400000)

extracted_actors = []
allowed_tconsts = set(target_movies_df['tconst'])

for piece in cast_reader:
    relevant_cast = piece[piece['tconst'].isin(allowed_tconsts) & piece['category'].isin(['actor', 'actress'])]
    if not relevant_cast.empty:
        extracted_actors.append(relevant_cast)
        
all_cast_df = pd.concat(extracted_actors)
sorted_cast = all_cast_df.sort_values(by=['tconst', 'ordering']).drop_duplicates(subset=['tconst', 'nconst'])

top_actors_grouped = sorted_cast.groupby('tconst').head(5).groupby('tconst')['nconst'].apply(list).reset_index()
top_actors_grouped.columns = ['tconst', 'lead_actors_ids']

target_movies_df = pd.merge(target_movies_df, top_actors_grouped, on='tconst', how='left')
print("Base dataset preparation complete!")

Merging ratings...
Sorting complete. Taking top 5000 movies for processing.
Fetching lead actors in chunks (Memory Safe)...
Base dataset preparation complete!


Web scraping the remaining columns from Wikipedia.

In [4]:
def clean_html_string(raw_val):
    if pd.isna(raw_val) or not raw_val: return None
    text_clean = re.sub(r'\[.*?\]', '', str(raw_val))
    return re.sub(r',\s*,', ',', text_clean.replace('\n', ', ')).strip(' ,')

def scrape_movie_details(m_title, m_year):
    year_formatted = str(int(float(m_year))) if pd.notna(m_year) else ""
    url_safe_title = urllib.parse.quote(str(m_title).strip().replace(' ', '_'))
    
    wiki_urls = [
        f"https://en.wikipedia.org/wiki/{url_safe_title}_({year_formatted}_film)" if year_formatted else "",
        f"https://en.wikipedia.org/wiki/{url_safe_title}_(film)",
        f"https://en.wikipedia.org/wiki/{url_safe_title}"
    ]
    
    extracted_info = {'Language': None, 'Country': None, 'budget': None, 'BoxOffice': None, 'plot': None}
    req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    for w_link in [lnk for lnk in wiki_urls if lnk]:
        try:
            response = requests.get(w_link, headers=req_headers, timeout=5)
            if response.status_code != 200: continue
            
            page_content = BeautifulSoup(response.text, 'html.parser')
            
            # Extraction from the Infobox
            wiki_infobox = page_content.find('table', class_=re.compile('infobox'))
            if wiki_infobox:
                for tr_row in wiki_infobox.find_all('tr'):
                    th_cell = tr_row.find('th')
                    td_cell = tr_row.find('td')
                    if th_cell and td_cell:
                        header_title = th_cell.text.lower()
                        cell_val = clean_html_string(td_cell.get_text(separator=', ', strip=True))
                        
                        if 'budget' in header_title: extracted_info['budget'] = cell_val
                        elif 'box office' in header_title: extracted_info['BoxOffice'] = cell_val
                        elif 'language' in header_title: extracted_info['Language'] = cell_val
                        # התיקון שפותר את הבעיה של האביר האפל - זיהוי של Country וגם Countries
                        elif 'countr' in header_title or 'origin' in header_title or 'nation' in header_title: 
                            extracted_info['Country'] = cell_val
            
            # Extraction of the Plot
            plot_div = page_content.find(id=re.compile('(?i)^(plot|synopsis|premise|story)$'))
            if plot_div and plot_div.parent:
                story_paragraphs = []
                next_node = plot_div.parent.find_next_sibling()
                
                while next_node and next_node.name != 'h2':
                    if next_node.name == 'p' and next_node.text.strip():
                        story_paragraphs.append(clean_html_string(next_node.text))
                    next_node = next_node.find_next_sibling()
                
                if story_paragraphs:
                    extracted_info['plot'] = " ".join(story_paragraphs)
                    
            # אם מצאנו עלילה או תקציב - זה כנראה העמוד הנכון ואין טעם להמשיך לבדוק לינקים נוספים לסרט הזה
            if extracted_info['plot'] or extracted_info['budget']:
                return extracted_info
                
        except:
            continue
            
    return extracted_info

In [5]:
collected_movie_data = []
processed_count = 0
total_to_process = len(target_movies_df)

print(f"Starting the FULL scraping loop for {total_to_process} movies (Wikipedia ONLY)...")

for idx, r_data in target_movies_df.iterrows():
    dict_row = r_data.to_dict() 
    
    wiki_res = scrape_movie_details(r_data['primaryTitle'], r_data['startYear'])
    dict_row.update(wiki_res)

    collected_movie_data.append(dict_row)
    processed_count += 1
    
    if processed_count % 500 == 0:
        print(f"  ...processed {processed_count} out of {total_to_process} movies...")
        pd.DataFrame(collected_movie_data).to_csv('wikipedia_backup_checkpoint.csv', index=False, encoding='utf-8-sig')
        
    time.sleep(0.5) 

uncleaned_df = pd.DataFrame(collected_movie_data)
print("\n--- Scraping Phase Finished! ---")

Starting the FULL scraping loop for 5000 movies (Wikipedia ONLY)...
  ...processed 500 out of 5000 movies...
  ...processed 1000 out of 5000 movies...
  ...processed 1500 out of 5000 movies...
  ...processed 2000 out of 5000 movies...
  ...processed 2500 out of 5000 movies...
  ...processed 3000 out of 5000 movies...
  ...processed 3500 out of 5000 movies...
  ...processed 4000 out of 5000 movies...
  ...processed 4500 out of 5000 movies...
  ...processed 5000 out of 5000 movies...

--- Scraping Phase Finished! ---


Creating a function that converts all budget and revenue amounts into a uniform numerical format of millions of dollars.

In [20]:
if 'uncleaned_df' in locals() and isinstance(uncleaned_df, pd.DataFrame):
    df_clean = uncleaned_df.copy()
    print(f"Successfully loaded 'uncleaned_df' from memory with {len(df_clean)} records.")
elif 'collected_movie_data' in locals() and isinstance(collected_movie_data, list):
    df_clean = pd.DataFrame(collected_movie_data)
    print(f"Successfully loaded 'collected_movie_data' from memory with {len(df_clean)} records.")
else:
    print("Error: Could not find 'uncleaned_df' or 'collected_movie_data' in memory. Please run the scraping cell first.")

def convert_to_usd_millions(money_str):
    """
    Parses financial strings, identifies global currencies, handles value ranges,
    and normalizes the output to Millions of USD.
    """
    if pd.isna(money_str) or isinstance(money_str, (int, float)):
        return money_str
       
    text = str(money_str).lower().replace(',', '')
   
    # Top 30 global currencies and their conversion rates to USD
    # Extract all numeric values from the text
    extracted_nums = [float(n) for n in re.findall(r'\d+\.\d+|\d+', text)]
   
    # Check for scale descriptors
    scale_words = ['million', 'billion', 'thousand', 'm', 'b', 'k', 'crore', 'lakh']
    has_scale_word = any(word in text for word in scale_words)
   
    valid_nums = []
    for n in extracted_nums:
        # Ignore standalone years (e.g., 1998) unless attached to a scale word
        if 1900 <= n <= 2050 and n.is_integer() and not has_scale_word:
            continue
        valid_nums.append(n)
   
    if not valid_nums:
        return np.nan
       
    # Calculate averag
    base_val = sum(valid_nums) / len(valid_nums)
   
    # Handle Indian numbering system (common in Bollywood entries)
    if 'crore' in text:
        base_val *= 10
    elif 'lakh' in text:
        base_val /= 10
       
    # Scale adjustments to reach Millions
    if 'billion' in text or ' b ' in text:
        base_val *= 1000
    elif 'thousand' in text or ' k ' in text:
        base_val /= 1000
    elif 'million' not in text and base_val >= 1000000:
        base_val /= 1000000
       
    return round(base_val * rate, 2)

if 'df_clean' in locals():
    print("Normalizing financial columns...")
    df_clean['budget'] = df_clean['budget'].apply(convert_to_usd_millions)
    df_clean['BoxOffice'] = df_clean['BoxOffice'].apply(convert_to_usd_millions)

    print("Formatting numeric types...")
    # Convert appropriate columns to Int64 to remove the ".0" artifact
    for col in ['startYear', 'runtimeMinutes', 'numVotes']:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')

    # Reorder columns logically for the final dataset
    target_columns = [
        'tconst', 'primaryTitle', 'startYear', 'Country', 'Language',
        'genres', 'runtimeMinutes', 'averageRating', 'numVotes',
        'lead_actors_ids', 'budget', 'BoxOffice', 'plot'
    ]

    # Keep only columns that exist
    existing_columns = [c for c in target_columns if c in df_clean.columns]
    df_final = df_clean[existing_columns]

    # Export final dataset
    output_filename = 'Movies_Final_Data.csv'
    df_final.to_csv(output_filename, index=False, na_rep='NaN', encoding='utf-8-sig')

    print(f"Process Complete! Final clean dataset saved to: '{output_filename}'")
    display(df_final[['primaryTitle', 'Country', 'Language', 'budget', 'BoxOffice']].head(10))

Successfully loaded 'uncleaned_df' from memory with 5000 records.
Normalizing financial columns...
Formatting numeric types...
Process Complete! Final clean dataset saved to: 'Movies_Final_Data.csv'


,primaryTitle,Country,Language,budget,BoxOffice
0,The Shawshank Redemption,United States,English,25.0,73.3
1,The Dark Knight,"United States, United Kingdom",English,185.0,1009.0
2,The Matrix,"United States, , Australia",English,63.0,467.8
3,The Godfather,United States,English,6.5,270.5
4,The Lord of the Rings: The Fellowship of the Ring,"New Zealand, United States",English,93.0,897.0
5,The Lord of the Rings: The Return of the King,"New Zealand, Germany, United States",English,94.0,1148.0
6,The Dark Knight Rises,"United States, United Kingdom",English,260.0,1115.0
7,The Lord of the Rings: The Two Towers,"New Zealand, United States",English,94.0,944.0
8,The Wolf of Wall Street,United States,English,100.0,407.0
9,The Silence of the Lambs,United States,English,19.0,272.7


Checking the percentage of missing values in each column.

In [21]:
df = pd.read_csv('Movies_Final_Data.csv')

columns_to_check = ['plot', 'Language', 'budget', 'Country', 'BoxOffice']

df_subset = df[columns_to_check]
missing_counts = df_subset.isnull().sum()
missing_percent = (df_subset.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percent
})

# הדפסת התוצאות, ממוינות מהאחוז הגבוה לנמוך
print(missing_summary.sort_values(by='Percentage (%)', ascending=False))

           Missing Count  Percentage (%)
budget              3699           73.98
BoxOffice           3489           69.78
plot                3228           64.56
Country             3211           64.22
Language            3203           64.06


Presentation of the final table.

In [22]:
# טעינת הקובץ שנוצר
df = pd.read_csv('Movies_Final_Data.csv')

# הצגת 5 השורות הראשונות
display(df.head())

,tconst,primaryTitle,startYear,Country,Language,genres,runtimeMinutes,averageRating,numVotes,lead_actors_ids,budget,BoxOffice,plot
0,tt0111161,The Shawshank Redemption,1994,United States,English,['Drama'],142,9.3,3186698,"['nm0000209', 'nm0000151', 'nm0348409', 'nm000...",25.0,73.3,"In 1947, Portland, Maine, banker Andy Dufresne..."
1,tt0468569,The Dark Knight,2008,"United States, United Kingdom",English,"['Action', 'Crime', 'Drama']",152,9.1,3165820,"['nm0000288', 'nm0005132', 'nm0001173', 'nm000...",185.0,1009.0,A gang of masked criminals rob a mafia-owned b...
2,tt0133093,The Matrix,1999,"United States, , Australia",English,"['Action', 'Sci-Fi']",136,8.7,2246428,"['nm0000206', 'nm0000401', 'nm0005251', 'nm091...",63.0,467.8,"In 1999, in an unnamed city, Thomas Anderson, ..."
3,tt0068646,The Godfather,1972,United States,English,"['Crime', 'Drama']",175,9.2,2225065,"['nm0000008', 'nm0000199', 'nm0001001', 'nm000...",6.5,270.5,Michael Corleone is a World War II veteran and...
4,tt0120737,The Lord of the Rings: The Fellowship of the Ring,2001,"New Zealand, United States",English,"['Adventure', 'Drama', 'Fantasy']",178,8.9,2205424,"['nm0000704', 'nm0005212', 'nm0089217', 'nm000...",93.0,897.0,"In the Second Age of Middle-earth, the lords o..."
